# 03 - OLS Biased Baseline

Estimate the naive OLS effect of PM2.5 on test scores. This is the 'before' — the biased estimate we improve upon with IV and RDD.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

PANEL_FILE = "analysis_panel.parquet"
if not (DATA_DIR / PANEL_FILE).exists():
    raise FileNotFoundError(
        "Analysis panel not found. Build it by running in order:\n"
        "  python scripts/download_epa_aqs.py --email EMAIL --key KEY\n"
        "  python scripts/download_hms_smoke.py\n"
        "  python scripts/download_seda.py  (manual — see instructions)\n"
        "  python src/merge/build_crosswalks.py\n"
        "  python src/ingest/epa_aqs.py\n"
        "  python src/ingest/seda.py\n"
        "  python src/exposure/smoke_instrument.py\n"
        "  python src/merge/build_panel.py"
    )

panel = pd.read_parquet(DATA_DIR / PANEL_FILE)
print(f"Panel: {panel.shape}")
print(f"Districts: {panel['leaid'].nunique()}")
print(f"Years: {sorted(panel['year'].dropna().unique())}")

In [ ]:
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS

## Model 1 - Bivariate OLS

In [ ]:
m1 = smf.ols("test_score_mean ~ pm25_annual_mean", data=panel).fit(cov_type="HC3")
print(f"Bivariate OLS: β = {m1.params['pm25_annual_mean']:.4f}")
print("Expected: near zero or positive (confounded upward by poverty)") 

## Model 2 - OLS with district and year FE

In [ ]:
panel_fe = panel.dropna(subset=["leaid","year","test_score_mean","pm25_annual_mean"]).copy()
panel_fe["year"] = panel_fe["year"].astype(int)
idx = panel_fe.set_index(["leaid","year"])

fe = PanelOLS(
    idx["test_score_mean"],
    idx[["pm25_annual_mean"]],
    entity_effects=True,
    time_effects=True,
).fit(cov_type="clustered", cluster_entity=True)

b_fe = fe.params["pm25_annual_mean"]
ci   = fe.conf_int().loc["pm25_annual_mean"]
print(f"TWFE: β = {b_fe:.4f}  95% CI [{ci['lower']:.4f}, {ci['upper']:.4f}]")
print("Better — but still confounded by time-varying poverty shocks")
print("IV (notebook 04) addresses remaining endogeneity")

## Why FE doesn't fully solve it

District FE removes time-invariant poverty. But time-varying shocks — a factory closing, a new highway, economic decline — affect both PM2.5 and school quality simultaneously. The smoke instrument fixes this by using only variation in PM2.5 that comes from upwind wildfire activity, which is plausibly exogenous to local economic conditions.